In [6]:
import os
import numpy as np
import pandas as pd
import librosa
from scipy.stats import kendalltau
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

print("Fase 3 Otimizada: Validação de Rigor Científico do Dataset")
print("=" * 80)

# =============================================================================
# BLOCO 1 — CONFIGURAÇÃO
# =============================================================================
DIR_AUDIO_DRY = "nsynth-test/audio"
DIR_AUDIO_WET = "dataset_fase2_final/audio"
FILE_METADATA = "dataset_fase2_final/metadata.csv"

df_meta = pd.read_csv(FILE_METADATA)

# =============================================================================
# BLOCO 2 — EXTRAÇÃO DE FEATURES AVANÇADA
# =============================================================================
def extrair_metrics_v2(audio_path):
    y, sr = librosa.load(audio_path, sr=None, mono=True)
    if len(y) < 100: return None # Ignora ficheiros corrompidos
    
    # Audio Features (AF)
    rms = np.mean(librosa.feature.rms(y=y))
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    # Flatness: Mede quão "tonal" ou "ruidoso" é o som (ideal para crisp/muffled)
    flatness = np.mean(librosa.feature.spectral_flatness(y=y))
    # Rolloff: Frequência abaixo da qual 85% da energia reside (ideal para bright/dark)
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
    
    return y, rms, centroid, flatness, rolloff, mfccs

# =============================================================================
# BLOCO 3 — PROCESSAMENTO
# =============================================================================
print("\nProcessando amostra de validação...")
notas_amostra = np.random.choice(df_meta['original_note'].unique(), size=400, replace=False)
df_valid = df_meta[df_meta['original_note'].isin(notas_amostra)]

resultados = []
for _, row in tqdm(df_valid.iterrows(), total=len(df_valid)):
    path_wet = os.path.join(DIR_AUDIO_WET, row['file_name'])
    res = extrair_metrics_v2(path_wet)
    if res:
        y, rms, cent, flat, roll, mfcc = res
        resultados.append({
            'keyword': row['keyword'],
            'intensity': row['intensity'],
            'original_note': row['original_note'],
            'centroid': cent,
            'flatness': flat,
            'rolloff': roll,
            'rms': rms,
            'mfccs': mfcc
        })

df_feat = pd.DataFrame(resultados)

# =============================================================================
# BLOCO 4 — MONOTONICIDADE (TAU DE KENDALL) COM FILTRO DE ENERGIA
# =============================================================================
print("\n" + "=" * 80)
print("1. TESTE DE MONOTONICIDADE (TAU DE KENDALL)")
print("-" * 80)

# Mapeamento físico-perceptual refinado
feature_alvo = {
    'bright': ('rolloff', 1),   # Sobe
    'dark':   ('rolloff', -1),  # Desce
    'crisp':  ('flatness', 1),  # Mais harmónicos/ruído
    'muffled':('flatness', -1), # Mais tonal/abafado
    'warm':   ('centroid', -1), # Foco em médios/graves
    'tinny':  ('centroid', 1),  # Foco em agudos metálicos
}

for kw, (feat, weight) in feature_alvo.items():
    df_kw = df_feat[df_feat['keyword'] == kw]
    if df_kw.empty: continue
    
    taus = []
    for nota in df_kw['original_note'].unique():
        df_n = df_kw[df_kw['original_note'] == nota].sort_values('intensity')
        if len(df_n) < 3: continue
        
        t, _ = kendalltau(df_n['intensity'], df_n[feat])
        if not np.isnan(t): taus.append(t * weight)
        
    if taus:
        print(f"  [{kw.upper():10s}] Tau: {np.mean(taus):.3f} (Ideal > 0.7)")

# =============================================================================
# BLOCO 5 — MMD PADRONIZADO (DISTÂNCIA DE CLUSTER)
# =============================================================================
print("\n" + "=" * 80)
print("2. MMD PADRONIZADO (DISTÂNCIA ENTRE OPOSTOS)")
print("-" * 80)

scaler = StandardScaler()
X_all_mfcc = scaler.fit_transform(np.stack(df_feat['mfccs'].values))

def mmd_rbf(X, Y, gamma=0.1):
    from sklearn.metrics.pairwise import rbf_kernel
    XX = rbf_kernel(X, X, gamma); YY = rbf_kernel(Y, Y, gamma); XY = rbf_kernel(X, Y, gamma)
    return XX.mean() + YY.mean() - 2 * XY.mean()

pares = [("bright", "dark"), ("warm", "tinny"), ("crisp", "muffled")]
for p1, p2 in pares:
    idx1 = df_feat[(df_feat['keyword'] == p1) & (df_feat['intensity'] == 1.0)].index
    idx2 = df_feat[(df_feat['keyword'] == p2) & (df_feat['intensity'] == 1.0)].index
    if len(idx1) > 0 and len(idx2) > 0:
        dist = mmd_rbf(X_all_mfcc[idx1], X_all_mfcc[idx2])
        print(f"  [{p1} vs {p2}] MMD: {dist:.4f} (Ideal > 0.1)")

# =============================================================================
# BLOCO 6 — BASELINE MLP COM MATRIZ DE CONFUSÃO
# =============================================================================
print("\n" + "=" * 80)
print("3. CLASSIFICADOR BASELINE (PROVA-DE-CONCEITO)")
print("-" * 80)

df_max = df_feat[df_feat['intensity'] == 1.0]
X = scaler.fit_transform(np.stack(df_max['mfccs'].values))
y = df_max['keyword'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000).fit(X_train, y_train)
acc = accuracy_score(y_test, clf.predict(X_test))

print(f"  Acurácia Final: {acc*100:.2f}% (Acaso: {100/len(np.unique(y)):.2f}%)")
print("\nValidação Concluída. Se Acurácia > 50% e Tau > 0.5, o dataset é robusto.")

Fase 3 Otimizada: Validação de Rigor Científico do Dataset

Processando amostra de validação...


100%|███████████████████████████████████████████████████████████████████████████████| 8000/8000 [01:44<00:00, 76.28it/s]



1. TESTE DE MONOTONICIDADE (TAU DE KENDALL)
--------------------------------------------------------------------------------
  [BRIGHT    ] Tau: -0.692 (Ideal > 0.7)
  [DARK      ] Tau: 0.042 (Ideal > 0.7)
  [CRISP     ] Tau: 0.268 (Ideal > 0.7)
  [MUFFLED   ] Tau: 0.768 (Ideal > 0.7)
  [WARM      ] Tau: 0.507 (Ideal > 0.7)
  [TINNY     ] Tau: -0.403 (Ideal > 0.7)

2. MMD PADRONIZADO (DISTÂNCIA ENTRE OPOSTOS)
--------------------------------------------------------------------------------
  [bright vs dark] MMD: 0.5466 (Ideal > 0.1)
  [warm vs tinny] MMD: 0.4532 (Ideal > 0.1)
  [crisp vs muffled] MMD: 0.3472 (Ideal > 0.1)

3. CLASSIFICADOR BASELINE (PROVA-DE-CONCEITO)
--------------------------------------------------------------------------------
  Acurácia Final: 65.62% (Acaso: 5.88%)

Validação Concluída. Se Acurácia > 50% e Tau > 0.5, o dataset é robusto.


In [2]:
!pip install librosa scikit-learn scipy

  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 11.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 4.5 MB/s  0:00:08m0:00:0100:01
Using cached audioread-3.1.0-py3-none-any.whl (23 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 3.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 4.0 MB/s  0:00:14m0:00:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [librosa]0/12 [numba]-learn]
